# Crime Map Inspection Notebook

Stateless flow: no widgets, no callback accumulation.
Run cells top-to-bottom each time.


In [4]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "crime_map").is_dir() and (candidate / "app").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root containing crime_map/ and app/.")

ROOT_DIR = find_repo_root(Path.cwd())
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

ROOT_DIR


WindowsPath('c:/Users/ncomati/Documents/GitHub/crime-map')

In [5]:
from crime_map import (
    get_supported_municipalities,
    get_bundle,
    filter_crime_by_date,
    compute_relative_rates,
    build_choropleth_map,
    reset_state,
)

# Hard reset so each run starts clean. Set clear_disk_cache=False if you do not want re-downloads.
reset_state(clear_disk_cache=True)

get_supported_municipalities()


['All Metro', 'Cambridge', 'Boston', 'Somerville']

In [7]:
city = "All Metro"
selected_macro = "Violent Crime"
start_date = None
end_date = None

bundle = get_bundle(city)
crime_df = filter_crime_by_date(bundle["crime"], start_date, end_date)
rates_df = compute_relative_rates(crime_df, bundle["population"])

if selected_macro not in rates_df.columns:
    selected_macro = "Violent Crime" if "Violent Crime" in rates_df.columns else rates_df.columns[0]

folium_map = build_choropleth_map(
    geo_df=bundle["geo"],
    rates_df=rates_df,
    population=bundle["population"],
    selected_macro=selected_macro,
    zoom_start=bundle["zoom"],
    population_year=bundle["population_year"],
)

folium_map